# Hands-on Workshop on Transformers — Version 1
## Build Transformer Blocks from Scratch + Train a Text Classifier

**Duration:** 2 hours  
**Audience:** B.Tech beginner/intermediate students  
**Dataset:** AG News text classification  
**Framework:** PyTorch  
**Works on:** Google Colab, Kaggle Notebook, and local Jupyter Notebook

This notebook follows the same flow as your Transformer PPT:

1. Tokenization
2. Input embeddings
3. Positional encodings
4. Query, Key, Value
5. Attention
6. Self-attention
7. Multi-head attention
8. Feed-forward network
9. Add & Norm
10. Encoder block
11. Linear + Softmax
12. Application: text classification

Version 1 is educational: students build the Transformer encoder blocks themselves.


# 0. Two-hour lab plan

| Time | Section | Goal |
|---:|---|---|
| 0–10 min | Setup and GPU check | Make notebook platform-safe |
| 10–25 min | Dataset and tokenization | Convert real text into tokens and IDs |
| 25–40 min | Embedding + positional encoding | Connect words with order |
| 40–60 min | Q, K, V and attention | Understand scaled dot-product attention |
| 60–75 min | Multi-head attention | Multiple attention views |
| 75–90 min | Transformer encoder block | Attention + FFN + Add & Norm |
| 90–110 min | Train classifier | Use real AG News dataset |
| 110–120 min | Visualization + tasks | Student experiments |


# 1. Platform-safe setup

For Colab: `Runtime → Change runtime type → GPU`  
For Kaggle: `Notebook Settings → Accelerator → GPU` and enable Internet  
For local Jupyter:

```bash
pip install torch datasets pandas scikit-learn matplotlib tqdm
```


In [ ]:
import sys, subprocess, importlib.util

packages = {
    "torch": "torch",
    "datasets": "datasets",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

missing = [pip_name for import_name, pip_name in packages.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All required packages are already installed.")


# 2. Imports, seed, and GPU check


In [ ]:
import os, re, math, random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# 3. Run mode

Use `lecture_fast` for live teaching. Use `high_gpu` if you have enough time and GPU memory.


In [ ]:
RUN_MODE = "lecture_fast"   # "debug", "lecture_fast", "high_gpu"

CONFIGS = {
    "debug": dict(train_samples=2000, val_samples=500, test_samples=1000,
                  max_vocab=10000, max_len=64, batch_size=64,
                  d_model=64, num_heads=4, num_layers=1, d_ff=128,
                  dropout=0.1, epochs=1, lr=3e-4),
    "lecture_fast": dict(train_samples=16000, val_samples=4000, test_samples=6000,
                         max_vocab=30000, max_len=96, batch_size=128,
                         d_model=128, num_heads=4, num_layers=2, d_ff=256,
                         dropout=0.15, epochs=3, lr=3e-4),
    "high_gpu": dict(train_samples=50000, val_samples=10000, test_samples=10000,
                     max_vocab=50000, max_len=128, batch_size=256,
                     d_model=256, num_heads=8, num_layers=4, d_ff=512,
                     dropout=0.2, epochs=5, lr=3e-4),
}
cfg = CONFIGS[RUN_MODE]
cfg


# 4. Load real dataset: AG News

AG News is a real-world news topic classification dataset.

Classes:

```text
0 = World
1 = Sports
2 = Business
3 = Sci/Tech
```

Relation with PPT: this is an **encoder-style Transformer application**.
The encoder reads the full input text and the linear + softmax layer predicts the class.


In [ ]:
def load_ag_news():
    try:
        from datasets import load_dataset
        ds = load_dataset("ag_news")
        train_df = pd.DataFrame(ds["train"])[["text", "label"]]
        test_df = pd.DataFrame(ds["test"])[["text", "label"]]
        return train_df, test_df, "Hugging Face ag_news"
    except Exception as e:
        print("Could not load AG News:", repr(e))
        print("Using tiny fallback dataset only to keep notebook executable.")
        examples = {
            0: ["The government announced a new international policy.", "Foreign leaders met for peace talks."],
            1: ["The team won the football match.", "The player scored in the final game."],
            2: ["The stock market closed higher.", "The company reported record profit."],
            3: ["Scientists released a new AI model.", "A new satellite was launched into orbit."],
        }
        rows = []
        for _ in range(1000):
            for label, texts in examples.items():
                for text in texts:
                    rows.append({"text": text, "label": label})
        df = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        split = int(0.8 * len(df))
        return df.iloc[:split].reset_index(drop=True), df.iloc[split:].reset_index(drop=True), "tiny fallback"

train_df, test_df, DATA_SOURCE = load_ag_news()

train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

val_df = train_df.iloc[:cfg["val_samples"]].reset_index(drop=True)
train_df = train_df.iloc[cfg["val_samples"]:cfg["val_samples"] + cfg["train_samples"]].reset_index(drop=True)
test_df = test_df.iloc[:cfg["test_samples"]].reset_index(drop=True)

label_names = ["World", "Sports", "Business", "Sci/Tech"]

print("Data source:", DATA_SOURCE)
print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
train_df.head()


# 5. Inspect real examples


In [ ]:
for i in range(5):
    print("=" * 100)
    print("Label:", label_names[int(train_df.loc[i, "label"])])
    print(train_df.loc[i, "text"])


# 6. Tokenization

PPT relation: **Tokenization**

The PPT example:

```text
I ate an apple → I, ate, an, apple, <eos>
```

Here we use a simple word-level tokenizer for teaching.


In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
CLS_TOKEN = "<CLS>"

def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9']+", " ", text)
    return text.strip().split()

def build_vocab(texts, max_vocab):
    counter = Counter()
    for text in tqdm(texts, desc="Building vocabulary"):
        counter.update(simple_tokenize(text))
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1, CLS_TOKEN: 2}
    for word, freq in counter.most_common(max_vocab - len(vocab)):
        vocab[word] = len(vocab)
    return vocab, counter

vocab, counter = build_vocab(train_df["text"].tolist(), cfg["max_vocab"])
id_to_word = {v: k for k, v in vocab.items()}

print("Vocabulary size:", len(vocab))
print("Most common tokens:", counter.most_common(20))


# 7. Convert tokens to IDs and create attention mask

PPT relation: **Input preparation**

The attention mask is `1` for real tokens and `0` for padding tokens.


In [ ]:
def encode_text(text, vocab, max_len):
    tokens = [CLS_TOKEN] + simple_tokenize(text)
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokens]
    ids = ids[:max_len]
    mask = [1] * len(ids)
    pad_len = max_len - len(ids)
    ids = ids + [vocab[PAD_TOKEN]] * pad_len
    mask = mask + [0] * pad_len
    return ids, mask

sample = "Apple releases a new AI chip for faster computers."
ids, mask = encode_text(sample, vocab, 12)
print("Text:", sample)
print("Tokens:", [CLS_TOKEN] + simple_tokenize(sample))
print("IDs:", ids)
print("Mask:", mask)


# 8. Dataset and DataLoader


In [ ]:
class NewsDataset(Dataset):
    def __init__(self, df, vocab, max_len):
        self.texts = df["text"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, mask = encode_text(self.texts[idx], self.vocab, self.max_len)
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.bool),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = NewsDataset(train_df, vocab, cfg["max_len"])
val_ds = NewsDataset(val_df, vocab, cfg["max_len"])
test_ds = NewsDataset(test_df, vocab, cfg["max_len"])

train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("label:", batch["label"].shape)


# 9. Input Embeddings

PPT relation: **Input Embeddings**

An embedding layer maps:

```text
token ID → vector of size d_model
```


In [ ]:
embedding_demo = nn.Embedding(len(vocab), cfg["d_model"], padding_idx=vocab[PAD_TOKEN])
demo_ids = batch["input_ids"][:2]
demo_emb = embedding_demo(demo_ids)
print("Input IDs shape:", demo_ids.shape)
print("Embedding output shape:", demo_emb.shape)


# 10. Positional Encoding

PPT relation: **Position Encodings**

Transformers process tokens in parallel, so we add position information.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

pos_demo = SinusoidalPositionalEncoding(cfg["d_model"], max_len=cfg["max_len"])
demo_with_pos = pos_demo(demo_emb)
print("Before:", demo_emb.shape)
print("After:", demo_with_pos.shape)


# 11. Visualize positional encoding


In [ ]:
pe_matrix = pos_demo.pe[0, :50, :32].detach().cpu().numpy()

plt.figure(figsize=(10, 5))
plt.imshow(pe_matrix, aspect="auto")
plt.colorbar(label="positional encoding value")
plt.xlabel("Embedding dimension")
plt.ylabel("Position")
plt.title("Sinusoidal Positional Encoding")
plt.tight_layout()
plt.show()


# 12. Scaled Dot-Product Attention

PPT relation: **Query, Key, Value** and **Attention**

```text
scores = QKᵀ / sqrt(d_k)
weights = softmax(scores)
output = weights × V
```


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

B, T, D = 2, 5, 16
Q, K, V = torch.randn(B, T, D), torch.randn(B, T, D), torch.randn(B, T, D)
out, weights = scaled_dot_product_attention(Q, K, V)

print("Q:", Q.shape, "K:", K.shape, "V:", V.shape)
print("Attention weights:", weights.shape)
print("Output:", out.shape)
print("First row sum:", weights[0, 0].sum().item())


# 13. Multi-Head Self-Attention

PPT relation: **Self Attention** and **Multi-Head Attention**

Self-attention means Q, K, and V come from the same input sequence.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, T, D = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        B, H, T, Hd = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, H * Hd)

    def forward(self, x, attention_mask=None, return_attention=False):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))
        mask = attention_mask.unsqueeze(1).unsqueeze(2) if attention_mask is not None else None
        context, weights = scaled_dot_product_attention(Q, K, V, mask)
        context = self.combine_heads(context)
        output = self.dropout(self.W_o(context))
        if return_attention:
            return output, weights
        return output

mha_demo = MultiHeadSelfAttention(cfg["d_model"], cfg["num_heads"])
demo_out, demo_attn = mha_demo(demo_with_pos, batch["attention_mask"][:2], return_attention=True)
print("MHA output:", demo_out.shape)
print("Attention weights:", demo_attn.shape)


# 14. Feed Forward, Add & Norm, Encoder Block


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attention_mask=None, return_attention=False):
        if return_attention:
            attn_out, weights = self.attn(x, attention_mask, return_attention=True)
        else:
            attn_out = self.attn(x, attention_mask)
            weights = None
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return (x, weights) if return_attention else x

block_demo = TransformerEncoderBlock(cfg["d_model"], cfg["num_heads"], cfg["d_ff"])
block_out, block_attn = block_demo(demo_with_pos, batch["attention_mask"][:2], return_attention=True)
print("Encoder block output:", block_out.shape)


# 15. Full Transformer Encoder Classifier

PPT relation:

```text
Tokenization → Embedding → Positional Encoding → Encoder Blocks → Linear → Softmax
```

We use the `<CLS>` token vector for classification.


In [ ]:
class TransformerTextClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model, num_heads, num_layers,
                 d_ff, max_len, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.positional_encoding = SinusoidalPositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask=None, return_attention=False):
        x = self.embedding(input_ids)
        x = self.positional_encoding(x)
        attentions = []
        for layer in self.layers:
            if return_attention:
                x, attn = layer(x, attention_mask, return_attention=True)
                attentions.append(attn)
            else:
                x = layer(x, attention_mask)
        cls_repr = x[:, 0, :]
        logits = self.classifier(self.dropout(cls_repr))
        return (logits, attentions) if return_attention else logits

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = TransformerTextClassifier(
    vocab_size=len(vocab),
    num_classes=4,
    d_model=cfg["d_model"],
    num_heads=cfg["num_heads"],
    num_layers=cfg["num_layers"],
    d_ff=cfg["d_ff"],
    max_len=cfg["max_len"],
    dropout=cfg["dropout"],
    pad_idx=vocab[PAD_TOKEN],
).to(DEVICE)

print(model)
print("Trainable parameters:", count_params(model))


# 16. Training and evaluation functions


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for batch in tqdm(loader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * input_ids.size(0)
        preds_all.extend(logits.argmax(dim=1).detach().cpu().tolist())
        labels_all.extend(labels.detach().cpu().tolist())
    return total_loss / len(loader.dataset), accuracy_score(labels_all, preds_all)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for batch in tqdm(loader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        total_loss += loss.item() * input_ids.size(0)
        preds_all.extend(logits.argmax(dim=1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())
    return total_loss / len(loader.dataset), accuracy_score(labels_all, preds_all), labels_all, preds_all


# 17. Train the Transformer classifier


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)

history = []
best_val_acc = 0.0
best_path = "best_transformer_from_scratch.pt"

for epoch in range(1, cfg["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

    history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                        val_loss=val_loss, val_acc=val_acc))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)

    print(f"Epoch {epoch}/{cfg['epochs']} | "
          f"Train loss {train_loss:.4f}, acc {train_acc:.4f} | "
          f"Val loss {val_loss:.4f}, acc {val_acc:.4f}")

history_df = pd.DataFrame(history)
history_df


# 18. Plot training history


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="train_acc")
plt.plot(history_df["epoch"], history_df["val_acc"], marker="o", label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Transformer from Scratch: Accuracy")
plt.legend()
plt.grid(True)
plt.show()


# 19. Test evaluation


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)

print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_acc, 4))
print(classification_report(y_true, y_pred, target_names=label_names))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))


# 20. Custom prediction


In [ ]:
@torch.no_grad()
def predict_news(text):
    model.eval()
    ids, mask = encode_text(text, vocab, cfg["max_len"])
    input_ids = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    attention_mask = torch.tensor([mask], dtype=torch.bool).to(DEVICE)
    logits = model(input_ids, attention_mask)
    probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred = int(np.argmax(probs))
    return {"text": text, "prediction": label_names[pred],
            "probabilities": {label_names[i]: float(probs[i]) for i in range(4)}}

examples = [
    "The football team won the final after a dramatic penalty shootout.",
    "The government announced a new policy after meeting foreign leaders.",
    "The company shares rose after strong quarterly earnings.",
    "Researchers developed a new machine learning system for medical diagnosis.",
]
for ex in examples:
    print("=" * 100)
    print(predict_news(ex))


# 21. Attention visualization


In [ ]:
@torch.no_grad()
def visualize_attention(text, layer_idx=0, head_idx=0):
    model.eval()
    ids, mask = encode_text(text, vocab, cfg["max_len"])
    tokens = [id_to_word.get(i, UNK_TOKEN) for i, m in zip(ids, mask) if m == 1]

    input_ids = torch.tensor([ids], dtype=torch.long).to(DEVICE)
    attention_mask = torch.tensor([mask], dtype=torch.bool).to(DEVICE)
    logits, attentions = model(input_ids, attention_mask, return_attention=True)

    attn = attentions[layer_idx][0, head_idx].detach().cpu().numpy()
    L = len(tokens)
    attn = attn[:L, :L]

    plt.figure(figsize=(8, 6))
    plt.imshow(attn, aspect="auto")
    plt.colorbar(label="attention weight")
    plt.xticks(range(L), tokens, rotation=45, ha="right")
    plt.yticks(range(L), tokens)
    plt.xlabel("Key tokens")
    plt.ylabel("Query tokens")
    plt.title(f"Attention map | layer {layer_idx}, head {head_idx}")
    plt.tight_layout()
    plt.show()

    print("Prediction:", label_names[int(logits.argmax(dim=1).item())])

visualize_attention("Researchers launched a new AI model for medical image analysis.", layer_idx=0, head_idx=0)


# 22. Student hands-on tasks

## Task 1: Change model capacity

Change:

```python
cfg["d_model"]
cfg["num_heads"]
cfg["num_layers"]
cfg["d_ff"]
```

Try `d_model=256`, `num_heads=8`, `num_layers=3`.

Observe parameter count, speed, and validation accuracy.

---

## Task 2: Explore attention heads

Run:

```python
visualize_attention("your sentence here", layer_idx=1, head_idx=2)
```

Question: do different heads attend to different words?

---

## Task 3: Convert this into binary classification

Make:

```text
Sports = 1
All other classes = 0
```

Retrain the model. Is binary classification easier than 4-class classification?
